# Phase 9 — Orchestration

**Objectifs:**
- [x] Databricks Workflows
- [x] Task Dependencies
- [x] Scheduling
- [x] Logs et Monitoring

**Livrable:** Automated Workflow

**Objectif:** exécuter le pipeline **automatiquement**, dans le bon ordre, chaque jour, avec relance en cas d'échec et suivi des exécutions.

Le Job est défini **en code** (*Infrastructure as Code*) dans `resources/lakehouse_job.yml` et déployé avec les **Databricks Asset Bundles** (`databricks.yml`).

## 0. Setup

In [1]:
from lakehouse_config import *
spark = get_spark()

from pyspark.sql import functions as F
import pandas as pd

import subprocess, os
import lakehouse_config
lakehouse_config._add_cli_to_path()   # rend la CLI databricks disponible


def sh(cmd):
    """Execute une commande shell a la racine du projet et affiche la sortie."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, encoding="utf-8", errors="replace")
    out = "\n".join(l for l in (r.stdout + r.stderr).splitlines() if "aitools" not in l)
    print(out.strip()[-4000:])
    print(f"-> code retour: {r.returncode}")
    return r.returncode

## 1. Databricks Workflows

Un **Job** = un ensemble de **tâches** (notebooks, scripts, SQL…) exécutées sur du compute Databricks (ici **Serverless**: pas de cluster à gérer).

Définition du Job:

In [2]:
print(open("resources/lakehouse_job.yml", encoding="utf-8").read())

# Phase 9 — Databricks Workflow (Job) du pipeline Lakehouse
# Deploiement : databricks bundle deploy -t dev
# Execution   : databricks bundle run lakehouse_pipeline -t dev

resources:
  jobs:
    lakehouse_pipeline:
      name: lakehouse-nyc-taxi-pipeline
      description: "Pipeline NYC Taxi: ingestion incrementale (Bronze + Silver MERGE) -> Gold -> Data Quality"

      # Execution quotidienne a 06:00 (heure du Maroc).
      # PAUSED par defaut pour ne pas consommer de compute : passer a UNPAUSED pour activer la planification.
      schedule:
        quartz_cron_expression: "0 0 6 * * ?"
        timezone_id: "Africa/Casablanca"
        pause_status: PAUSED

      max_concurrent_runs: 1          # jamais 2 executions en parallele (evite les doublons)
      timeout_seconds: 3600

      tasks:
        - task_key: incremental_ingestion
          description: "Nouveaux fichiers -> Bronze (append + log) -> Silver (MERGE)"
          notebook_task:
            notebook_path: "../Phase 07 — In

## 2. Task Dependencies

```
┌──────────────────────────┐     ┌──────────────────┐     ┌────────────────┐
│ incremental_ingestion    │ ──► │ gold_analytics   │ ──► │ data_quality   │
│ Phase 07 (Bronze+Silver) │     │ Phase 05 (Gold)  │     │ Phase 06 (DQ)  │
└──────────────────────────┘     └──────────────────┘     └────────────────┘
```

- `depends_on`: une tâche ne démarre que si la précédente a **réussi**.
- Si `data_quality` lève `DataQualityError`, le run est marqué **FAILED** → alerte.
- `max_retries: 1` sur l'ingestion: une erreur réseau temporaire est relancée automatiquement.
- `max_concurrent_runs: 1`: deux runs ne peuvent pas écrire en même temps (évite les doublons).

In [3]:
import yaml
job = yaml.safe_load(open("resources/lakehouse_job.yml", encoding="utf-8"))["resources"]["jobs"]["lakehouse_pipeline"]
pd.DataFrame([{
    "task": t["task_key"],
    "depends_on": ", ".join(d["task_key"] for d in t.get("depends_on", [])) or "-",
    "notebook": t["notebook_task"]["notebook_path"].replace("../", ""),
    "retries": t.get("max_retries", 0),
} for t in job["tasks"]])

,task,depends_on,notebook,retries
0,incremental_ingestion,-,Phase 07 — Incremental Processing.ipynb,1
1,gold_analytics,incremental_ingestion,Phase 05 — Gold Analytics.ipynb,0
2,data_quality,gold_analytics,Phase 06 — Data Quality.ipynb,0


## 3. Scheduling

| Paramètre | Valeur | Signification |
|---|---|---|
| `quartz_cron_expression` | `0 0 6 * * ?` | Tous les jours à 06:00 |
| `timezone_id` | `Africa/Casablanca` | Fuseau horaire |
| `pause_status` | `PAUSED` | Planification **désactivée** par défaut (évite de consommer du compute). Passer à `UNPAUSED` puis redéployer pour l'activer |

Syntaxe Quartz: `secondes minutes heures jour_du_mois mois jour_de_la_semaine`. Exemples: `0 0 */6 * * ?` (toutes les 6 h), `0 30 7 ? * MON-FRI` (07:30 en semaine).

### 3.1 Validation et déploiement du bundle

In [4]:
sh('databricks bundle validate -t dev -p "marouane mrf"')

Name: Lakehouse-Databricks
Target: dev
Workspace:
  Host: https://dbc-8e6d5a22-8ef2.cloud.databricks.com
  User: marwane2001morfi@gmail.com
  Path: /Workspace/Users/marwane2001morfi@gmail.com/.bundle/Lakehouse-Databricks/dev

Validation OK!
-> code retour: 0


0

In [5]:
# Deploie le Job dans le workspace (mode development: nom prefixe [dev <user>])
sh('databricks bundle deploy -t dev -p "marouane mrf"')

Uploading bundle files to /Workspace/Users/marwane2001morfi@gmail.com/.bundle/Lakehouse-Databricks/dev/files...
Files: 5 uploaded, 5 deleted
Resources: 0 created, 0 changed, 0 deleted, 1 unchanged
-> code retour: 0


0

Pour lancer une exécution manuelle (consomme du compute Serverless):
```bash
databricks bundle run lakehouse_pipeline -t dev -p "marouane mrf"
```
Dans l'UI: **Jobs & Pipelines** → `[dev ...] lakehouse-nyc-taxi-pipeline` → **Run now**.

## 4. Logs et Monitoring
### 4.1 État du Job et de ses exécutions (Databricks SDK)

In [6]:
w = get_workspace_client()
jobs = [j for j in w.jobs.list(name=None) if "lakehouse-nyc-taxi-pipeline" in (j.settings.name or "")]

if not jobs:
    print("Job non deploye -> executer la cellule 'bundle deploy'.")
for j in jobs:
    s = j.settings
    print(f"Job : {s.name} (id {j.job_id})")
    print(f"Schedule: {s.schedule.quartz_cron_expression} {s.schedule.timezone_id} -> {s.schedule.pause_status.value}")
    runs = list(w.jobs.list_runs(job_id=j.job_id, limit=10))
    if not runs:
        print("Aucune execution pour l'instant.")
    for r in runs:
        print(f"  run {r.run_id} | {r.state.life_cycle_state.value} | {r.state.result_state.value if r.state.result_state else '-'}"
              f" | {r.run_page_url}")

Job : [dev marwane2001morfi] lakehouse-nyc-taxi-pipeline (id 901835922738402)
Schedule: 0 0 6 * * ? Africa/Casablanca -> PAUSED
Aucune execution pour l'instant.


### 4.2 Monitoring métier (tables `ops`)

In [7]:
# Fichiers ingeres par jour et statut
spark.sql(f"""
    SELECT date(ingested_at) AS jour, status, COUNT(*) AS fichiers, SUM(row_count) AS lignes
    FROM {T_INGESTION_LOG} GROUP BY ALL ORDER BY jour DESC
""").show()

+----------+-------+--------+-------+
|      jour| status|fichiers| lignes|
+----------+-------+--------+-------+
|2026-11-08|SUCCESS|       2|5972150|
+----------+-------+--------+-------+



In [8]:
# Qualite des donnees par execution
spark.sql(f"""
    SELECT run_id, MIN(run_ts) AS run_ts,
           SUM(CASE WHEN status = 'PASS' THEN 1 ELSE 0 END) AS pass,
           SUM(CASE WHEN status <> 'PASS' AND severity = 'warning'  THEN 1 ELSE 0 END) AS warnings,
           SUM(CASE WHEN status <> 'PASS' AND severity = 'critical' THEN 1 ELSE 0 END) AS critical
    FROM {T_DQ_RESULTS} WHERE run_id <> 'demo'
    GROUP BY run_id ORDER BY run_ts DESC LIMIT 10
""").show(truncate=False)

+------------------------------------+--------------------------+----+--------+--------+
|run_id                              |run_ts                    |pass|warnings|critical|
+------------------------------------+--------------------------+----+--------+--------+
|69a5fe94-b911-42c8-8176-b78df38ebf2c|2026-11-08 19:14:33.739476|11  |2       |0       |
|ca1c24b6-73dd-4522-9f63-255276dd1f14|2026-11-08 19:06:06.932599|11  |2       |0       |
+------------------------------------+--------------------------+----+--------+--------+



In [9]:
# Fraicheur des donnees: derniere ingestion et derniere course disponible
spark.sql(f"""
    SELECT (SELECT MAX(ingested_at) FROM {T_INGESTION_LOG})   AS derniere_ingestion,
           (SELECT MAX(pickup_datetime) FROM {T_SILVER_TRIPS}) AS derniere_course,
           (SELECT MAX(date) FROM {T_GOLD_DAILY_KPIS})         AS dernier_jour_gold
""").show(truncate=False)

+--------------------------+-------------------+-----------------+
|derniere_ingestion        |derniere_course    |dernier_jour_gold|
+--------------------------+-------------------+-----------------+
|2026-11-08 19:08:27.508162|2024-02-29 23:59:58|2024-02-29       |
+--------------------------+-------------------+-----------------+



### 4.3 Où trouver les logs
- **Run d'un job**: Jobs & Pipelines → job → onglet **Runs** → cliquer un run → chaque tâche affiche la **sortie du notebook** (cellules exécutées, erreurs, durée).
- **Historique des requêtes**: **Query History** → Query Profile (plans, temps par opérateur).
- **Tables système**: `system.lakeflow.job_run_timeline`, `system.lakeflow.job_task_run_timeline` (durées, statuts sur la durée).
- **Alertes**: `email_notifications` / `webhook_notifications` dans le YAML (`on_failure`, `on_duration_warning_threshold_exceeded`).

## Conclusion — Automated Workflow
- Pipeline **Incremental → Gold → Data Quality** défini **en code** (YAML) et versionné avec Git.
- **Dépendances** entre tâches, **retry** automatique, **un seul run à la fois**.
- **Planification** quotidienne (Quartz cron, fuseau Africa/Casablanca).
- **Monitoring** technique (runs du Job via SDK / UI) et **métier** (`ops.ingestion_log`, `ops.dq_results`, fraîcheur des données).